In [1]:
print("Here begins the BN models notebook.")

Here begins the BN models notebook.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
df_bn = pd.read_csv("../../data/processed/world_happiness_bn_clean.csv")

df_bn.head()

,Country,Happiness_Score,GDP_per_Capita,Social_Support,Healthy_Life_Expectancy,Freedom,Generosity,Corruption_Perception,Unemployment_Rate,Education_Index,Population,Urbanization_Rate,Life_Satisfaction,Public_Trust,Mental_Health_Index,Income_Inequality,Public_Health_Expenditure,Climate_Index,Work_Life_Balance,Internet_Access,Crime_Rate,Political_Stability,Employment_Rate,Year_bin
0,China,low,high,medium,medium,low,low,very_high,high,very_low,very_high,very_high,very_high,low,high,high,very_high,medium,very_high,high,very_high,low,low,2020
1,UK,medium,medium,very_high,low,very_high,low,very_high,very_high,high,high,low,low,high,low,high,low,low,very_high,very_high,very_high,high,high,2015
2,Brazil,low,high,very_low,low,very_low,medium,medium,very_high,very_high,medium,low,low,low,high,low,low,very_low,medium,medium,low,very_high,medium,2005
3,France,medium,medium,high,very_high,very_high,high,high,very_low,low,very_high,very_high,low,high,very_low,very_high,low,very_high,medium,very_high,medium,medium,very_low,2015
4,China,very_high,medium,very_low,very_low,high,medium,very_high,low,very_high,very_high,very_high,medium,medium,low,low,high,medium,very_low,medium,high,very_low,very_low,2020


In [2]:
# Inspect shape, data types, and missing values
print(df_bn.shape)
print(df_bn.dtypes)

print("\nMissing values per column:")
print(df_bn.isna().sum().sort_values(ascending=False))

(4000, 24)
Country                        str
Happiness_Score                str
GDP_per_Capita                 str
Social_Support                 str
Healthy_Life_Expectancy        str
Freedom                        str
Generosity                     str
Corruption_Perception          str
Unemployment_Rate              str
Education_Index                str
Population                     str
Urbanization_Rate              str
Life_Satisfaction              str
Public_Trust                   str
Mental_Health_Index            str
Income_Inequality              str
Public_Health_Expenditure      str
Climate_Index                  str
Work_Life_Balance              str
Internet_Access                str
Crime_Rate                     str
Political_Stability            str
Employment_Rate                str
Year_bin                     int64
dtype: object

Missing values per column:
Country                      0
Happiness_Score              0
GDP_per_Capita               0
Social_Support

In [3]:
# Convert all columns to string so pgmpy treats them as discrete states
# Missing values are preserved as NaN
for col in df_bn.columns:
    df_bn[col] = df_bn[col].astype("string")

print(df_bn.dtypes)

Country                      string
Happiness_Score              string
GDP_per_Capita               string
Social_Support               string
Healthy_Life_Expectancy      string
Freedom                      string
Generosity                   string
Corruption_Perception        string
Unemployment_Rate            string
Education_Index              string
Population                   string
Urbanization_Rate            string
Life_Satisfaction            string
Public_Trust                 string
Mental_Health_Index          string
Income_Inequality            string
Public_Health_Expenditure    string
Climate_Index                string
Work_Life_Balance            string
Internet_Access              string
Crime_Rate                   string
Political_Stability          string
Employment_Rate              string
Year_bin                     string
dtype: object


In [4]:
# Check number of states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn.columns,
    "Num_States": [df_bn[col].nunique(dropna=True) for col in df_bn.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
0,Country,10
1,Happiness_Score,5
2,GDP_per_Capita,5
3,Social_Support,5
4,Healthy_Life_Expectancy,5
5,Freedom,5
6,Generosity,5
7,Corruption_Perception,5
8,Unemployment_Rate,5
9,Education_Index,5


In [5]:
# Create BN modeling dataset
df_bn_model = df_bn.copy()

# Drop Country to avoid high-cardinality country identity dominating the graph
df_bn_model = df_bn_model.drop(columns=["Country"], errors="ignore")

print(df_bn_model.shape)
print(df_bn_model.columns.tolist())

(4000, 23)
['Happiness_Score', 'GDP_per_Capita', 'Social_Support', 'Healthy_Life_Expectancy', 'Freedom', 'Generosity', 'Corruption_Perception', 'Unemployment_Rate', 'Education_Index', 'Population', 'Urbanization_Rate', 'Life_Satisfaction', 'Public_Trust', 'Mental_Health_Index', 'Income_Inequality', 'Public_Health_Expenditure', 'Climate_Index', 'Work_Life_Balance', 'Internet_Access', 'Crime_Rate', 'Political_Stability', 'Employment_Rate', 'Year_bin']


In [6]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pgmpy.estimators import HillClimbSearch, ExpertKnowledge, BayesianEstimator
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

In [7]:
# Learn an unconstrained Bayesian Network structure
hc = HillClimbSearch(df_bn_model)

learned_dag = hc.estimate(
    scoring_method="k2",
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag.nodes()))
print("Number of edges:", len(learned_dag.edges()))
print("\nLearned edges:")
print(sorted(learned_dag.edges()))

Number of nodes: 23
Number of edges: 0

Learned edges:
[]


In [8]:
# Convert the learned graph into a discrete Bayesian Network
# Important: add all columns as nodes, including isolated ones
bn_unconstrained = DiscreteBayesianNetwork()
bn_unconstrained.add_nodes_from(df_bn_model.columns)
bn_unconstrained.add_edges_from(learned_dag.edges())

# Fit CPDs using Bayesian parameter estimation
bn_unconstrained.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Model check:", bn_unconstrained.check_model())

Model check: True


In [9]:
# Inspect direct parents of Life_Satisfaction
if "Life_Satisfaction" in bn_unconstrained.nodes():
    print("Parents of Life_Satisfaction in unconstrained BN:")
    print(list(bn_unconstrained.predecessors("Life_Satisfaction")))

# Inspect CPD metadata for the target variable
if "Life_Satisfaction" in bn_unconstrained.nodes():
    cpd_ls = bn_unconstrained.get_cpds("Life_Satisfaction")
    print("\nCPD state names for Life_Satisfaction:")
    print(cpd_ls.state_names)


Parents of Life_Satisfaction in unconstrained BN:
[]

CPD state names for Life_Satisfaction:
{'Life_Satisfaction': ['high', 'low', 'medium', 'very_high', 'very_low']}


In [11]:
# Build a restricted BN with expert constraints
# The goal is to prevent implausible directions and encourage a more interpretable structure
# for Life_Satisfaction in this macro-level dataset.

forbidden_edges = [
    # Life satisfaction should not cause macro indicators
    ("Life_Satisfaction", "GDP_per_Capita"),
    ("Life_Satisfaction", "Social_Support"),
    ("Life_Satisfaction", "Healthy_Life_Expectancy"),
    ("Life_Satisfaction", "Freedom"),
    ("Life_Satisfaction", "Generosity"),
    ("Life_Satisfaction", "Corruption_Perception"),
    ("Life_Satisfaction", "Unemployment_Rate"),
    ("Life_Satisfaction", "Education_Index"),
    ("Life_Satisfaction", "Population"),
    ("Life_Satisfaction", "Urbanization_Rate"),
    ("Life_Satisfaction", "Public_Trust"),
    ("Life_Satisfaction", "Mental_Health_Index"),
    ("Life_Satisfaction", "Income_Inequality"),
    ("Life_Satisfaction", "Public_Health_Expenditure"),
    ("Life_Satisfaction", "Climate_Index"),
    ("Life_Satisfaction", "Work_Life_Balance"),
    ("Life_Satisfaction", "Internet_Access"),
    ("Life_Satisfaction", "Crime_Rate"),
    ("Life_Satisfaction", "Political_Stability"),
    ("Life_Satisfaction", "Employment_Rate"),
    ("Life_Satisfaction", "Happiness_Score"),
    ("Life_Satisfaction", "Year_bin"),

    # Time should not be caused by current indicators
    ("GDP_per_Capita", "Year_bin"),
    ("Social_Support", "Year_bin"),
    ("Healthy_Life_Expectancy", "Year_bin"),
    ("Freedom", "Year_bin"),
    ("Generosity", "Year_bin"),
    ("Corruption_Perception", "Year_bin"),
    ("Unemployment_Rate", "Year_bin"),
    ("Education_Index", "Year_bin"),
    ("Population", "Year_bin"),
    ("Urbanization_Rate", "Year_bin"),
    ("Public_Trust", "Year_bin"),
    ("Mental_Health_Index", "Year_bin"),
    ("Income_Inequality", "Year_bin"),
    ("Public_Health_Expenditure", "Year_bin"),
    ("Climate_Index", "Year_bin"),
    ("Work_Life_Balance", "Year_bin"),
    ("Internet_Access", "Year_bin"),
    ("Crime_Rate", "Year_bin"),
    ("Political_Stability", "Year_bin"),
    ("Employment_Rate", "Year_bin"),
    ("Happiness_Score", "Year_bin"),
    ("Life_Satisfaction", "Year_bin"),

    # Optional: Happiness score should not cause the underlying macro drivers
    ("Happiness_Score", "GDP_per_Capita"),
    ("Happiness_Score", "Social_Support"),
    ("Happiness_Score", "Healthy_Life_Expectancy"),
    ("Happiness_Score", "Freedom"),
    ("Happiness_Score", "Public_Trust"),
    ("Happiness_Score", "Mental_Health_Index"),
]

In [12]:
expert_knowledge = ExpertKnowledge(
    forbidden_edges=forbidden_edges
)

hc_restricted = HillClimbSearch(df_bn_model)

learned_dag_restricted = hc_restricted.estimate(
    scoring_method="k2",
    expert_knowledge=expert_knowledge,
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag_restricted.nodes()))
print("Number of edges:", len(learned_dag_restricted.edges()))
print("\nRestricted BN edges:")
print(sorted(learned_dag_restricted.edges()))

Number of nodes: 23
Number of edges: 0

Restricted BN edges:
[]


In [13]:
# Fit the restricted BN
# Important: add all columns as nodes so isolated variables are preserved
bn_restricted = DiscreteBayesianNetwork()
bn_restricted.add_nodes_from(df_bn_model.columns)
bn_restricted.add_edges_from(learned_dag_restricted.edges())

bn_restricted.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Restricted model check:", bn_restricted.check_model())

Restricted model check: True


In [14]:
# Inspect direct parents of Life_Satisfaction in the restricted BN
if "Life_Satisfaction" in bn_restricted.nodes():
    print("Parents of Life_Satisfaction in restricted BN:")
    print(list(bn_restricted.predecessors("Life_Satisfaction")))

    print("\nChildren of Life_Satisfaction in restricted BN:")
    print(list(bn_restricted.successors("Life_Satisfaction")))

Parents of Life_Satisfaction in restricted BN:
[]

Children of Life_Satisfaction in restricted BN:
[]


In [15]:
# Create a smaller BN dataset with the most important macro-level variables
selected_cols = [
    "Life_Satisfaction",
    "GDP_per_Capita",
    "Social_Support",
    "Healthy_Life_Expectancy",
    "Freedom",
    "Mental_Health_Index",
    "Public_Trust",
    "Work_Life_Balance",
    "Year_bin"
]

df_bn_small = df_bn_model[selected_cols].copy()

print(df_bn_small.shape)
print(df_bn_small.columns.tolist())
df_bn_small.head()

(4000, 9)
['Life_Satisfaction', 'GDP_per_Capita', 'Social_Support', 'Healthy_Life_Expectancy', 'Freedom', 'Mental_Health_Index', 'Public_Trust', 'Work_Life_Balance', 'Year_bin']


,Life_Satisfaction,GDP_per_Capita,Social_Support,Healthy_Life_Expectancy,Freedom,Mental_Health_Index,Public_Trust,Work_Life_Balance,Year_bin
0,very_high,high,medium,medium,low,high,low,very_high,2020
1,low,medium,very_high,low,very_high,low,high,very_high,2015
2,low,high,very_low,low,very_low,high,low,medium,2005
3,low,medium,high,very_high,very_high,very_low,high,medium,2015
4,medium,medium,very_low,very_low,high,low,medium,very_low,2020


In [16]:
# Check number of states per variable in the reduced dataset
cardinality_small = pd.DataFrame({
    "Variable": df_bn_small.columns,
    "Num_States": [df_bn_small[col].nunique(dropna=True) for col in df_bn_small.columns]
}).sort_values("Num_States", ascending=False)

cardinality_small

,Variable,Num_States
0,Life_Satisfaction,5
1,GDP_per_Capita,5
2,Social_Support,5
3,Healthy_Life_Expectancy,5
4,Freedom,5
5,Mental_Health_Index,5
6,Public_Trust,5
7,Work_Life_Balance,5
8,Year_bin,4


In [17]:
# Build a theory-guided BN for the reduced world happiness dataset
# This is more appropriate than fully data-driven structure learning for a small macro dataset.

manual_edges_small = [
    ("Year_bin", "GDP_per_Capita"),
    ("Year_bin", "Healthy_Life_Expectancy"),
    ("Year_bin", "Mental_Health_Index"),
    ("Year_bin", "Work_Life_Balance"),

    ("GDP_per_Capita", "Life_Satisfaction"),
    ("Social_Support", "Life_Satisfaction"),
    ("Healthy_Life_Expectancy", "Life_Satisfaction"),
    ("Freedom", "Life_Satisfaction"),
    ("Mental_Health_Index", "Life_Satisfaction"),
    ("Public_Trust", "Life_Satisfaction"),
    ("Work_Life_Balance", "Life_Satisfaction"),
]

bn_small_manual = DiscreteBayesianNetwork()
bn_small_manual.add_nodes_from(df_bn_small.columns)
bn_small_manual.add_edges_from(manual_edges_small)

print("Number of nodes:", len(bn_small_manual.nodes()))
print("Number of edges:", len(bn_small_manual.edges()))
print("\nManual edges:")
print(sorted(bn_small_manual.edges()))

Number of nodes: 9
Number of edges: 11

Manual edges:
[('Freedom', 'Life_Satisfaction'), ('GDP_per_Capita', 'Life_Satisfaction'), ('Healthy_Life_Expectancy', 'Life_Satisfaction'), ('Mental_Health_Index', 'Life_Satisfaction'), ('Public_Trust', 'Life_Satisfaction'), ('Social_Support', 'Life_Satisfaction'), ('Work_Life_Balance', 'Life_Satisfaction'), ('Year_bin', 'GDP_per_Capita'), ('Year_bin', 'Healthy_Life_Expectancy'), ('Year_bin', 'Mental_Health_Index'), ('Year_bin', 'Work_Life_Balance')]


In [18]:
# Fit the theory-guided BN
bn_small_manual.fit(
    df_bn_small,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Manual model check:", bn_small_manual.check_model())

Manual model check: True


In [19]:
# Inspect direct parents and children of Life_Satisfaction
if "Life_Satisfaction" in bn_small_manual.nodes():
    print("Parents of Life_Satisfaction in manual BN:")
    print(list(bn_small_manual.predecessors("Life_Satisfaction")))

    print("\nChildren of Life_Satisfaction in manual BN:")
    print(list(bn_small_manual.successors("Life_Satisfaction")))

Parents of Life_Satisfaction in manual BN:
['GDP_per_Capita', 'Social_Support', 'Healthy_Life_Expectancy', 'Freedom', 'Mental_Health_Index', 'Public_Trust', 'Work_Life_Balance']

Children of Life_Satisfaction in manual BN:
[]


In [20]:
# Create inference engine for the manual BN
infer_small_manual = VariableElimination(bn_small_manual)  # type: ignore

In [21]:
# Query 1:
# Probability distribution of Life_Satisfaction under favorable macro conditions
q_small_manual_1 = infer_small_manual.query(
    variables=["Life_Satisfaction"],
    evidence={  # type: ignore
        "GDP_per_Capita": "very_high",
        "Social_Support": "very_high",
        "Healthy_Life_Expectancy": "very_high",
        "Freedom": "very_high"
    },
    show_progress=False
)

print(q_small_manual_1)

+------------------------------+--------------------------+
| Life_Satisfaction            |   phi(Life_Satisfaction) |
+==============================+==========================+
| Life_Satisfaction(high)      |                   0.2066 |
+------------------------------+--------------------------+
| Life_Satisfaction(low)       |                   0.1904 |
+------------------------------+--------------------------+
| Life_Satisfaction(medium)    |                   0.2064 |
+------------------------------+--------------------------+
| Life_Satisfaction(very_high) |                   0.2061 |
+------------------------------+--------------------------+
| Life_Satisfaction(very_low)  |                   0.1904 |
+------------------------------+--------------------------+


In [22]:
# Query 2:
# Probability distribution of Life_Satisfaction under adverse macro conditions
q_small_manual_2 = infer_small_manual.query(
    variables=["Life_Satisfaction"],
    evidence={  # type: ignore
        "GDP_per_Capita": "very_low",
        "Social_Support": "very_low",
        "Healthy_Life_Expectancy": "very_low",
        "Freedom": "very_low"
    },
    show_progress=False
)

print(q_small_manual_2)

+------------------------------+--------------------------+
| Life_Satisfaction            |   phi(Life_Satisfaction) |
+==============================+==========================+
| Life_Satisfaction(high)      |                   0.1952 |
+------------------------------+--------------------------+
| Life_Satisfaction(low)       |                   0.1949 |
+------------------------------+--------------------------+
| Life_Satisfaction(medium)    |                   0.2030 |
+------------------------------+--------------------------+
| Life_Satisfaction(very_high) |                   0.2034 |
+------------------------------+--------------------------+
| Life_Satisfaction(very_low)  |                   0.2035 |
+------------------------------+--------------------------+


In [23]:
# Query 3:
# Compare Life_Satisfaction under strong vs weak psychosocial/institutional conditions
q_small_manual_3_good = infer_small_manual.query(
    variables=["Life_Satisfaction"],
    evidence={  # type: ignore
        "Mental_Health_Index": "very_high",
        "Public_Trust": "very_high",
        "Work_Life_Balance": "very_high"
    },
    show_progress=False
)

q_small_manual_3_bad = infer_small_manual.query(
    variables=["Life_Satisfaction"],
    evidence={  # type: ignore
        "Mental_Health_Index": "very_low",
        "Public_Trust": "very_low",
        "Work_Life_Balance": "very_low"
    },
    show_progress=False
)

print("Manual BN - high mental health, trust, and work-life balance:")
print(q_small_manual_3_good)

print("\nManual BN - low mental health, trust, and work-life balance:")
print(q_small_manual_3_bad)

Manual BN - high mental health, trust, and work-life balance:
+------------------------------+--------------------------+
| Life_Satisfaction            |   phi(Life_Satisfaction) |
+==============================+==========================+
| Life_Satisfaction(high)      |                   0.2032 |
+------------------------------+--------------------------+
| Life_Satisfaction(low)       |                   0.1991 |
+------------------------------+--------------------------+
| Life_Satisfaction(medium)    |                   0.2000 |
+------------------------------+--------------------------+
| Life_Satisfaction(very_high) |                   0.1982 |
+------------------------------+--------------------------+
| Life_Satisfaction(very_low)  |                   0.1995 |
+------------------------------+--------------------------+

Manual BN - low mental health, trust, and work-life balance:
+------------------------------+--------------------------+
| Life_Satisfaction            |   p

In [24]:
# Summarize binned Life_Satisfaction
def summarize_life_satisfaction_bins(query_result):
    probs = dict(zip(
        [str(state) for state in query_result.state_names["Life_Satisfaction"]],
        query_result.values
    ))

    low = probs.get("very_low", 0) + probs.get("low", 0)
    medium = probs.get("medium", 0)
    high = probs.get("high", 0) + probs.get("very_high", 0)

    return pd.Series({
        "Low": low,
        "Medium": medium,
        "High": high
    })

print("Manual BN - favorable macro conditions:")
print(summarize_life_satisfaction_bins(q_small_manual_1))

print("\nManual BN - adverse macro conditions:")
print(summarize_life_satisfaction_bins(q_small_manual_2))

print("\nManual BN - high mental health / trust / balance:")
print(summarize_life_satisfaction_bins(q_small_manual_3_good))

print("\nManual BN - low mental health / trust / balance:")
print(summarize_life_satisfaction_bins(q_small_manual_3_bad))

Manual BN - favorable macro conditions:
Low       0.380855
Medium    0.206423
High      0.412721
dtype: float64

Manual BN - adverse macro conditions:
Low       0.398409
Medium    0.203034
High      0.398557
dtype: float64

Manual BN - high mental health / trust / balance:
Low       0.398583
Medium    0.200009
High      0.401408
dtype: float64

Manual BN - low mental health / trust / balance:
Low       0.402567
Medium    0.199027
High      0.398406
dtype: float64


In [25]:
# Define a simpler, interpretable BN structure
edges_reduced = [
    ("GDP_per_Capita", "Life_Satisfaction"),
    ("Social_Support", "Life_Satisfaction"),
    ("Healthy_Life_Expectancy", "Life_Satisfaction"),
    ("Freedom", "Life_Satisfaction"),
]

bn_reduced = DiscreteBayesianNetwork(edges_reduced)

print("Reduced BN edges:")
print(edges_reduced)

Reduced BN edges:
[('GDP_per_Capita', 'Life_Satisfaction'), ('Social_Support', 'Life_Satisfaction'), ('Healthy_Life_Expectancy', 'Life_Satisfaction'), ('Freedom', 'Life_Satisfaction')]


In [26]:
# Fit CPDs using Bayesian estimation
bn_reduced.fit(
    df_bn_small,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Model check:", bn_reduced.check_model())

Model check: True


In [27]:
print("Parents of Life_Satisfaction:")
print(list(bn_reduced.predecessors("Life_Satisfaction")))

Parents of Life_Satisfaction:
['GDP_per_Capita', 'Social_Support', 'Healthy_Life_Expectancy', 'Freedom']


In [28]:
infer_reduced = VariableElimination(bn_reduced) # type: ignore

In [33]:
q_good = infer_reduced.query(
    variables=["Life_Satisfaction"],
    evidence={ # type: ignore
        "GDP_per_Capita": "very_high",
        "Freedom": "very_high",
        "Social_Support": "very_high",
        "Healthy_Life_Expectancy": "very_high",
    },
    show_progress=False
)

print("Reduced BN - favorable conditions:")
print(q_good)

Reduced BN - favorable conditions:
+------------------------------+--------------------------+
| Life_Satisfaction            |   phi(Life_Satisfaction) |
+==============================+==========================+
| Life_Satisfaction(high)      |                   0.3330 |
+------------------------------+--------------------------+
| Life_Satisfaction(low)       |                   0.0005 |
+------------------------------+--------------------------+
| Life_Satisfaction(medium)    |                   0.3330 |
+------------------------------+--------------------------+
| Life_Satisfaction(very_high) |                   0.3330 |
+------------------------------+--------------------------+
| Life_Satisfaction(very_low)  |                   0.0005 |
+------------------------------+--------------------------+
